#Task 1

In [6]:
import pandas as pd
data = pd.read_csv('/content/drive/MyDrive/loanapproval.csv')
print(data.head())

   applicant_id  age  gender marital_status  annual_income  loan_amount  \
0             1   59    Male       Divorced         100073         7169   
1             2   49    Male        Married         112197        23556   
2             3   35    Male       Divorced          84429        27052   
3             4   63  Female         Single         124195        11313   
4             5   28  Female        Married          81627        13315   

   credit_score  num_dependents  existing_loans_count employment_status  \
0           793               1                     1        Unemployed   
1           789               0                     2          Employed   
2           372               1                     4        Unemployed   
3           808               3                     4     Self-employed   
4           689               0                     1        Unemployed   

   loan_approved  
0              1  
1              1  
2              0  
3              1  
4  

In [ ]:
# Check for missing values in 'loan_amount' and 'credit_score'
print("Missing values before imputation:")
print(data[['loan_amount', 'credit_score']].isnull().sum())

# Impute missing values with the median for 'loan_amount' and 'credit_score'
# Using .copy() to avoid SettingWithCopyWarning, ensuring changes are made on a copy of the slice.
data_imputed = data.copy()
# Updated to avoid FutureWarning by reassigning the column instead of using inplace=True
data_imputed['loan_amount'] = data_imputed['loan_amount'].fillna(data_imputed['loan_amount'].median())
data_imputed['credit_score'] = data_imputed['credit_score'].fillna(data_imputed['credit_score'].median())

print("\nMissing values after imputation:")
print(data_imputed[['loan_amount', 'credit_score']].isnull().sum())

# Display the first few rows of the updated DataFrame to show the effect of imputation
print("\nDataFrame head after imputation:")
display(data_imputed.head())

Missing values before imputation:
loan_amount     0
credit_score    0
dtype: int64

Missing values after imputation:
loan_amount     0
credit_score    0
dtype: int64

DataFrame head after imputation:


,applicant_id,age,gender,marital_status,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count,employment_status,loan_approved
0,1,59,Male,Divorced,100073,7169,793,1,1,Unemployed,1
1,2,49,Male,Married,112197,23556,789,0,2,Employed,1
2,3,35,Male,Divorced,84429,27052,372,1,4,Unemployed,0
3,4,63,Female,Single,124195,11313,808,3,4,Self-employed,1
4,5,28,Female,Married,81627,13315,689,0,1,Unemployed,1


In [ ]:
# Map 0 to 'Rejected' and 1 to 'Approved' in the 'loan_approved' column
# We will apply this transformation on the data_imputed DataFrame.
loan_status_mapping = {0: 'Rejected', 1: 'Approved'}
data_imputed['loan_status'] = data_imputed['loan_approved'].map(loan_status_mapping)

# Display the updated DataFrame with the new 'loan_status' column
print("DataFrame head with encoded loan status:")
display(data_imputed[['loan_approved', 'loan_status']].head())

DataFrame head with encoded loan status:


,loan_approved,loan_status
0,1,Approved
1,1,Approved
2,0,Rejected
3,1,Approved
4,1,Approved


In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# Drop 'applicant_id' as it's an identifier and not a predictive feature.
# Drop 'loan_approved' from X as it's the target variable.
# Drop 'loan_status' as it's an encoded version of the target for display.
X = data_imputed.drop(columns=['applicant_id', 'loan_approved', 'loan_status'])
y = data_imputed['loan_approved']

# Perform one-hot encoding for categorical features in X
# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True) # drop_first avoids multicollinearity

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print("Shapes of the split datasets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Display first few rows of X_train and y_train
print("\nX_train head:")
display(X_train.head())
print("\ny_train head:")
display(y_train.head())

Shapes of the split datasets:
X_train shape: (800, 11)
X_test shape: (200, 11)
y_train shape: (800,)
y_test shape: (200,)

X_train head:


,age,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count,gender_Male,marital_status_Married,marital_status_Single,employment_status_Self-employed,employment_status_Unemployed
29,47,83646,26300,428,1,4,True,False,False,False,False
535,56,92954,26542,301,4,4,False,True,False,False,True
695,48,134604,35581,329,2,3,True,True,False,False,True
557,59,28472,48463,397,4,4,True,False,True,False,True
836,55,39086,43811,329,1,2,True,True,False,True,False



y_train head:


,loan_approved
29,1
535,0
695,0
557,0
836,1


In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical features to be scaled (excluding applicant_id, loan_approved, and loan_status)
numerical_features = ['age', 'annual_income', 'loan_amount', 'credit_score', 'num_dependents', 'existing_loans_count']

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the numerical features
data_scaled_numerical = scaler.fit_transform(data_imputed[numerical_features])

# Create a DataFrame from the scaled numerical features
data_scaled = pd.DataFrame(data_scaled_numerical, columns=numerical_features)

# Optionally, you can add back the non-numerical columns to the scaled DataFrame
# For now, let's just display the scaled numerical features

print("Head of DataFrame with scaled numerical features:")
display(data_scaled.head())

Head of DataFrame with scaled numerical features:


,age,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count
0,1.307840,0.482301,-1.566427,1.391620,-0.689042,-0.737500
1,0.514489,0.805362,-0.287825,1.365954,-1.394305,-0.029726
2,-0.596204,0.065444,-0.015048,-1.309808,-0.689042,1.385820
3,1.625181,1.125066,-1.243090,1.487871,0.721484,1.385820
4,-1.151550,-0.009219,-1.086883,0.724284,-1.394305,-0.737500


#Task 2

In [11]:
import pandas as pd
import re

# Load the dataset
data = pd.read_csv('/content/drive/MyDrive/SocialMediaTop100.csv', encoding='latin1')

print('Original DataFrame Info:')
data.info()

# Clean column names by stripping whitespace
data.columns = data.columns.str.strip()

# Display cleaned column names
print('\nCleaned Column Names:')
print(data.columns)

# Check for missing values
print('\nMissing values before cleaning:')
print(data.isnull().sum())

# Impute missing values for 'FOLLOWERS' and 'POSTS' with the median
for col in ['FOLLOWERS', 'POSTS']:
    if data[col].isnull().any():
        median_val = data[col].median()
        data[col] = data[col].fillna(median_val)

print('\nMissing values after imputation:')
print(data.isnull().sum())

# Clean 'PROFILE' column: remove special unicode characters and strip whitespace
data['PROFILE'] = data['PROFILE'].apply(lambda x: re.sub(r'<U\+\w+>', '', x).strip() if isinstance(x, str) else x)

# Clean 'Platform' column: strip whitespace
data['Platform'] = data['Platform'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Convert 'FOLLOWERS' and 'POSTS' to integer type after imputation
data['FOLLOWERS'] = data['FOLLOWERS'].astype(int)
data['POSTS'] = data['POSTS'].astype(int)

print('\nDataFrame head after cleaning:')
display(data.head())

print('\nDataFrame info after cleaning:')
data.info()

Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   N          500 non-null    int64  
 1   PROFILE    500 non-null    object 
 2   FOLLOWERS  500 non-null    float64
 3   POSTS      400 non-null    float64
 4   Platform   500 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 19.7+ KB

Cleaned Column Names:
Index(['N', 'PROFILE', 'FOLLOWERS', 'POSTS', 'Platform'], dtype='object')

Missing values before cleaning:
N              0
PROFILE        0
FOLLOWERS      0
POSTS        100
Platform       0
dtype: int64

Missing values after imputation:
N            0
PROFILE      0
FOLLOWERS    0
POSTS        0
Platform     0
dtype: int64

DataFrame head after cleaning:


,N,PROFILE,FOLLOWERS,POSTS,Platform
0,1,Instagram,539446645,7202,Instagram
1,2,Cristiano Ronaldo,473864939,3338,Instagram
2,3,Kylie,364542529,6935,Instagram
3,4,Leo Messi,355790796,890,Instagram
4,5,Selena Gomez,341579063,1828,Instagram



DataFrame info after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   N          500 non-null    int64 
 1   PROFILE    500 non-null    object
 2   FOLLOWERS  500 non-null    int64 
 3   POSTS      500 non-null    int64 
 4   Platform   500 non-null    object
dtypes: int64(3), object(2)
memory usage: 19.7+ KB


#Task 3

In [12]:
data = pd.read_csv('/content/drive/MyDrive/weather_forecast_data.csv')
print(data)

      Temperature   Humidity  Wind_Speed  Cloud_Cover     Pressure     Rain
0       23.720338  89.592641    7.335604    50.501694  1032.378759     rain
1       27.879734  46.489704    5.952484     4.990053   992.614190  no rain
2       25.069084  83.072843    1.371992    14.855784  1007.231620  no rain
3       23.622080  74.367758    7.050551    67.255282   982.632013     rain
4       20.591370  96.858822    4.643921    47.676444   980.825142  no rain
...           ...        ...         ...          ...          ...      ...
2495    21.791602  45.270902   11.807192    55.044682  1017.686181  no rain
2496    27.558479  46.481744   10.884915    39.715133  1008.590961  no rain
2497    28.108274  43.817178    2.897128    75.842952   999.119187  no rain
2498    14.789275  57.908105    2.374717     2.378743  1046.501875  no rain
2499    26.554356  97.101517   18.563084    81.357508  1001.729176  no rain

[2500 rows x 6 columns]


### Preprocessing `weather_forecast_data`

The current `weather_forecast_data` DataFrame does not contain a date column. Therefore, it's not possible to directly convert a date column to datetime format or create new features like 'season' and 'week_number' from it.

However, I will proceed with the other preprocessing steps:
1.  Handle missing values in 'Temperature' and 'Humidity'.
2.  Convert the 'Rain' column from categorical to numerical.

In [13]:
# Check for missing values before imputation
print("Missing values before imputation:")
print(data[['Temperature', 'Humidity']].isnull().sum())

# Impute missing values with the median for 'Temperature' and 'Humidity'
data['Temperature'] = data['Temperature'].fillna(data['Temperature'].median())
data['Humidity'] = data['Humidity'].fillna(data['Humidity'].median())

# Check for missing values after imputation
print("\nMissing values after imputation:")
print(data[['Temperature', 'Humidity']].isnull().sum())

Missing values before imputation:
Temperature    0
Humidity       0
dtype: int64

Missing values after imputation:
Temperature    0
Humidity       0
dtype: int64


In [14]:
# Convert 'Rain' column from categorical to numerical (0 for 'no rain', 1 for 'rain')
data['Rain_encoded'] = data['Rain'].apply(lambda x: 1 if x == 'rain' else 0)

print("\nHead of DataFrame with 'Rain_encoded' column:")
display(data[['Rain', 'Rain_encoded']].head())

print("\nValue counts for 'Rain_encoded':")
print(data['Rain_encoded'].value_counts())


Head of DataFrame with 'Rain_encoded' column:


,Rain,Rain_encoded
0,rain,1
1,no rain,0
2,no rain,0
3,rain,1
4,no rain,0



Value counts for 'Rain_encoded':
Rain_encoded
0    2186
1     314
Name: count, dtype: int64


In [15]:
print("\nFinal DataFrame head after preprocessing:")
display(data.head())


Final DataFrame head after preprocessing:


,Temperature,Humidity,Wind_Speed,Cloud_Cover,Pressure,Rain,Rain_encoded
0,23.720338,89.592641,7.335604,50.501694,1032.378759,rain,1
1,27.879734,46.489704,5.952484,4.990053,992.614190,no rain,0
2,25.069084,83.072843,1.371992,14.855784,1007.231620,no rain,0
3,23.622080,74.367758,7.050551,67.255282,982.632013,rain,1
4,20.591370,96.858822,4.643921,47.676444,980.825142,no rain,0


#Task 4

In [16]:
# @title
data = pd.read_csv('/content/drive/MyDrive/IMDB-Movie-Data.csv')
print(data.head)

<bound method NDFrame.head of      Rank                    Title                     Genre  \
0       1  Guardians of the Galaxy   Action,Adventure,Sci-Fi   
1       2               Prometheus  Adventure,Mystery,Sci-Fi   
2       3                    Split           Horror,Thriller   
3       4                     Sing   Animation,Comedy,Family   
4       5            Suicide Squad  Action,Adventure,Fantasy   
..    ...                      ...                       ...   
995   996     Secret in Their Eyes       Crime,Drama,Mystery   
996   997          Hostel: Part II                    Horror   
997   998   Step Up 2: The Streets       Drama,Music,Romance   
998   999             Search Party          Adventure,Comedy   
999  1000               Nine Lives     Comedy,Family,Fantasy   

                                           Description              Director  \
0    A group of intergalactic criminals are forced ...            James Gunn   
1    Following clues to the origin of man

In [17]:
# Display basic information about the DataFrame to identify missing values and data types
print("Original DataFrame Info:")
data.info()

# Check for missing values
print("\nMissing values before imputation:")
print(data.isnull().sum())

Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Rank                1000 non-null   int64  
 1   Title               1000 non-null   object 
 2   Genre               1000 non-null   object 
 3   Description         1000 non-null   object 
 4   Director            1000 non-null   object 
 5   Actors              1000 non-null   object 
 6   Year                1000 non-null   int64  
 7   Runtime (Minutes)   1000 non-null   int64  
 8   Rating              1000 non-null   float64
 9   Votes               1000 non-null   int64  
 10  Revenue (Millions)  872 non-null    float64
 11  Metascore           936 non-null    float64
dtypes: float64(3), int64(4), object(5)
memory usage: 93.9+ KB

Missing values before imputation:
Rank                    0
Title                   0
Genre                   0
Descrip

In [18]:
# Impute missing 'Rating' values with the median
data['Rating'] = data['Rating'].fillna(data['Rating'].median())

# Impute missing 'Genre' values with 'Unknown'
data['Genre'] = data['Genre'].fillna('Unknown')

# Check for missing values after imputation
print("\nMissing values after imputation:")
print(data[['Rating', 'Genre']].isnull().sum())


Missing values after imputation:
Rating    0
Genre     0
dtype: int64


In [19]:
# Remove duplicate movie entries
duplicates_before = data.duplicated().sum()
data.drop_duplicates(inplace=True)
duplicates_after = data.duplicated().sum()

print(f"Number of duplicate entries before removal: {duplicates_before}")
print(f"Number of duplicate entries after removal: {duplicates_after}")

Number of duplicate entries before removal: 0
Number of duplicate entries after removal: 0


In [20]:
from sklearn.preprocessing import MinMaxScaler

# Scale 'Rating' values between 0 and 1
scaler = MinMaxScaler()
data['Rating_scaled'] = scaler.fit_transform(data[['Rating']])

print("\nHead of DataFrame with scaled 'Rating_scaled' column:")
display(data[['Rating', 'Rating_scaled']].head())


Head of DataFrame with scaled 'Rating_scaled' column:


,Rating,Rating_scaled
0,8.1,0.873239
1,7.0,0.718310
2,7.3,0.760563
3,7.2,0.746479
4,6.2,0.605634


The 'Genre' column contains multiple genres separated by commas. To encode this, we'll first split the genres and then apply one-hot encoding.

In [21]:
import numpy as np

# Split 'Genre' column into individual genres
data['Genre'] = data['Genre'].apply(lambda x: x.split(',') if isinstance(x, str) else [x])

# Create a list of all unique genres
all_genres = np.unique([genre for sublist in data['Genre'] for genre in sublist])

# Create one-hot encoded columns for each genre
for genre in all_genres:
    data[genre] = data['Genre'].apply(lambda genres: 1 if genre in genres else 0)

# Drop the original 'Genre' column if desired
data.drop('Genre', axis=1, inplace=True)

print("\nDataFrame head after one-hot encoding 'Genre' and scaling 'Rating':")
display(data.head())


DataFrame head after one-hot encoding 'Genre' and scaling 'Rating':


,Rank,Title,Description,Director,Actors,Year,Runtime (Minutes),Rating,Votes,Revenue (Millions),...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,1,Guardians of the Galaxy,A group of intergalactic criminals are forced ...,James Gunn,"Chris Pratt, Vin Diesel, Bradley Cooper, Zoe S...",2014,121,8.1,757074,333.13,...,0,0,0,0,0,1,0,0,0,0
1,2,Prometheus,"Following clues to the origin of mankind, a te...",Ridley Scott,"Noomi Rapace, Logan Marshall-Green, Michael Fa...",2012,124,7.0,485820,126.46,...,0,0,0,1,0,1,0,0,0,0
2,3,Split,Three girls are kidnapped by a man with a diag...,M. Night Shyamalan,"James McAvoy, Anya Taylor-Joy, Haley Lu Richar...",2016,117,7.3,157606,138.12,...,1,0,0,0,0,0,0,1,0,0
3,4,Sing,"In a city of humanoid animals, a hustling thea...",Christophe Lourdelet,"Matthew McConaughey,Reese Witherspoon, Seth Ma...",2016,108,7.2,60545,270.32,...,0,0,0,0,0,0,0,0,0,0
4,5,Suicide Squad,A secret government agency recruits some of th...,David Ayer,"Will Smith, Jared Leto, Margot Robbie, Viola D...",2016,123,6.2,393727,325.02,...,0,0,0,0,0,0,0,0,0,0
